[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/75_video_tubelet_patchify_solution.ipynb)

# 🔴 Solution: Video Tubelet Patchify

Reference solution for `video_tubelet_patchify`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

class VideoTubeletPatcher:
    def __init__(self, tubelet_size: int, patch_size: int):
        self.tubelet_size = tubelet_size
        self.patch_size = patch_size

    def patchify(self, video: torch.Tensor) -> torch.Tensor:
        B, T, C, H, W = video.shape
        t, p = self.tubelet_size, self.patch_size
        if T % t != 0 or H % p != 0 or W % p != 0:
            raise ValueError("T/H/W must be divisible by tubelet_size/patch_size")
        nt, nh, nw = T // t, H // p, W // p
        x = video.view(B, nt, t, C, nh, p, nw, p)
        x = x.permute(0, 1, 4, 6, 2, 3, 5, 7).contiguous()
        return x.view(B, nt * nh * nw, t * C * p * p)

    def unpatchify(self, tokens: torch.Tensor, video_shape: tuple) -> torch.Tensor:
        B, T, C, H, W = video_shape
        t, p = self.tubelet_size, self.patch_size
        nt, nh, nw = T // t, H // p, W // p
        expected_dim = t * C * p * p
        if tokens.shape != (B, nt * nh * nw, expected_dim):
            raise ValueError("tokens do not match video_shape")
        x = tokens.view(B, nt, nh, nw, t, C, p, p)
        x = x.permute(0, 1, 4, 5, 2, 6, 3, 7).contiguous()
        return x.view(B, T, C, H, W)


In [ ]:
# Verify
video = torch.arange(1 * 4 * 3 * 4 * 4).float().view(1, 4, 3, 4, 4)
patcher = VideoTubeletPatcher(2, 2)
tokens = patcher.patchify(video)
print(tokens.shape)
print(torch.equal(patcher.unpatchify(tokens, video.shape), video))


In [ ]:
# Run judge
from torch_judge import check
check('video_tubelet_patchify')
